# Этап 2: google/gemma-4-E4B-it (`gemma4_e4b`)

Фаза 3, отбор моделей: Gemma 4 E4B (Apache 2.0), другое семейство. Модель грузится целиком, с башнями зрения и звука; LoRA - только на языковую часть. Из 7,9 млрд параметров 3,5 млрд - встраивания по слоям, они не квантуются: память на T4 покажет ячейка 4.

Цель: научить модель переводить описание схемы на русском в нетлист для Micro-Cap. Метод - QLoRA: веса модели заморожены в 4-битном виде, обучается только малый добавочный слой (LoRA), поэтому хватает бесплатного GPU.

**Перед началом:** Среда выполнения → Сменить среду выполнения → GPU (T4). Ячейки запускаются сверху вниз, вывод виден прямо здесь. Всё важное пишется на Google Drive в папку `mcnl/<RUN>`. Если сессия оборвалась, запустите ячейки заново: обучение продолжится с последней контрольной точки, готовые шаги пропускаются, а ответы модели дописываются с места обрыва.

## 1. Среда
Ставим библиотеки и проверяем, какой GPU выдан.

In [ ]:
%cd /content
!pip -q install -U transformers peft accelerate bitsandbytes sentencepiece "protobuf>=7.36"
!nvidia-smi --query-gpu=name,memory.total --format=csv
# токенизатор SentencePiece (YandexGPT) читается через protobuf: transformers собран под 7.x, в Colab стоит 6.x
import google.protobuf
print('protobuf', google.protobuf.__version__, '- ниже 7: Среда выполнения -> Перезапустить сеанс и снова эту ячейку'
      if int(google.protobuf.__version__.split('.')[0]) < 7 else '')

## 2. Drive и код проекта
Drive попросит разрешение. Ячейка проверяет свободное место: нужно не меньше 3 ГБ (контрольные точки, адаптер, ответы). Удалённые файлы остаются в корзине Drive и занимают место, пока корзину не очистить. Код берётся из GitHub, внизу видно, какой коммит запущен.
Если репозиторий закрытый, нужен токен только на чтение: значок ключа слева → секрет `GITHUB_TOKEN`, доступ для ноутбука включён. В текст ячеек и в чат токен не вставлять.

In [ ]:
import os, subprocess
os.chdir('/content')  # при повторном запуске текущая папка - /content/repo, которую ячейка сейчас удалит
from google.colab import drive, userdata
drive.mount('/content/drive')


def drive_free(need=3):
    # Свободно на Drive, ГБ. Colab пишет на Drive через кэш: когда место кончилось, ячейки идут без ошибок, а файлы
    # на Drive не попадают (08.11.2026, Phi-4-mini). Удалённые контрольные точки лежат в корзине Drive и занимают место.
    import shutil
    free = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
    print(f'Свободно на Drive: {free:.1f} ГБ')
    if free < need:
        raise SystemExit(f'На Drive меньше {need} ГБ: очистите корзину Drive (drive.google.com -> Корзина -> '
                         'Очистить корзину) и запустите ячейку снова. Иначе результаты не сохранятся.')
    return free


drive_free()
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None  # открытый репозиторий: токен не нужен
REPO = 'github.com/xdstk/lang.cir.spice.mc12'
!rm -rf /content/repo
done = subprocess.run(['git', 'clone', '-q', '--depth', '1', f'https://x-access-token:{token}@{REPO}' if token else f'https://{REPO}', '/content/repo'],
                      capture_output=True, text=True, env={'GIT_TERMINAL_PROMPT': '0', 'PATH': '/usr/bin:/bin:/usr/local/bin'})
if done.returncode:
    raise SystemExit('Код не скачан: ' + done.stderr.replace(token or '<нет>', '***').strip())
!git -C /content/repo remote set-url origin https://{REPO}
%cd /content/repo
!git log --oneline -1

## 3. Настройки запуска
Настройки этого запуска; ноутбуки других моделей отличаются только ими.

In [ ]:
MODEL = 'google/gemma-4-E4B-it'
RUN = 'gemma4_e4b'     # имя запуска: папка на Drive
SUBSET = 2000          # вложенное подмножество train v2: 1000, 2000 или 3000
LIMIT = 0              # сколько записей взять из подмножества; 0 - все
FORMAT = 'netlist'     # netlist или ir
EPOCHS = 2
BATCH = 2              # записей за проход; шаг оптимизации всегда 8 записей
ACCUM = 8 // BATCH
SEED = 1               # порядок записей и начальные веса LoRA
OUT = f'/content/drive/MyDrive/mcnl/{RUN}'
!mkdir -p {OUT}
!python src/mcnl/training.py --data data/dataset/v2 --subset {SUBSET} --limit {LIMIT} --format {FORMAT} --out out/training/{RUN}

## 4. Проба модели
Шаблон чата: в начале - инструкция (у YandexGPT она внутри запроса пользователя), в конце запроса - начало ответа ассистента (у Qwen3.5 с пустым блоком `<think></think>`: рассуждение выключено). Учиться модель должна только нетлисту и знаку конца реплики (`<|im_end|>`, `<turn|>` или `</s>`). Затем 10 шагов обучения во временную папку: проверка, что модель обучается на T4, и оценка времени полного запуска.

In [ ]:
import sys
sys.path.insert(0, 'src')
from transformers import AutoTokenizer
from mcnl import training
tok = AutoTokenizer.from_pretrained(MODEL)
recs = training.read_jsonl(f'out/training/{RUN}/train.jsonl')
end = training.end_of_turn(tok)
e = training.encode(tok, recs[0]['messages'], 10 ** 6, end)
print(tok.decode(e['input_ids'])[-300:])
print('УЧИТСЯ:', repr(tok.decode([t for t, l in zip(e['input_ids'], e['labels']) if l != -100])))
lens = sorted(len(training.encode(tok, r['messages'], 10 ** 6, end)['input_ids']) for r in recs)
print(f'токенов в примере: медиана {lens[len(lens) // 2]}, максимум {lens[-1]}')
!rm -rf /content/probe
!python colab/train_lora.py --model {MODEL} --data out/training/{RUN} --out /content/probe --epochs {EPOCHS} --seed {SEED} --batch {BATCH} --accum {ACCUM} --max-steps 10

## 5. Модель без обучения
Первая загрузка скачивает веса. Ответы на 30 записей dev - точка отсчёта: что умеет модель до обучения, только по инструкции.

In [ ]:
!python colab/predict.py --model {MODEL} --format {FORMAT} --records data/dataset/v2/dev.jsonl --limit 30 --out {OUT}/untrained_dev.jsonl --name untrained

## 6. Обучение
Здесь видны полоса прогресса, потери `loss` по шагам и `eval_loss` по эпохам. `loss` должен падать. Если он стоит на месте или становится `nan`, обучение идёт неправильно.

In [ ]:
!python colab/train_lora.py --model {MODEL} --data out/training/{RUN} --out {OUT} --epochs {EPOCHS} --seed {SEED} --batch {BATCH} --accum {ACCUM}
drive_free(1)

## 7. График обучения
Сохраняется на Drive как `loss.png`: годится для текста работы.

In [ ]:
import json, matplotlib.pyplot as plt
run = json.load(open(f'{OUT}/run.json', encoding='utf-8'))
tr = [(x['step'], x['loss']) for x in run['log'] if 'loss' in x]
ev = [(x['step'], x['eval_loss']) for x in run['log'] if 'eval_loss' in x]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label='обучающая выборка')
if ev:
    plt.plot(*zip(*ev), 'o-', label='dev')
plt.xlabel('шаг обучения'); plt.ylabel('потери'); plt.legend(); plt.grid(alpha=0.3)
plt.title(RUN); plt.savefig(f'{OUT}/loss.png', dpi=200, bbox_inches='tight'); plt.show()
print(f"Время обучения: {run['train_seconds'] / 60:.1f} мин (все сессии), пик памяти GPU: {run['gpu_peak_gb']:.1f} ГБ, GPU: {run['gpu']}")
print('Отброшено длинных примеров (train, dev):', run['dropped_long'])
print('Версии:', run['versions'])

## 8. Ответы обученной модели
Три набора: 30 записей из обучающей выборки (модель должна справиться почти полностью), весь dev (150) и запросы C01, которых модель не видела и которые написаны не шаблонами.

In [ ]:
A = f'--model {MODEL} --adapter {OUT}/adapter --format {FORMAT} --name {RUN}'
!python colab/predict.py {A} --records data/dataset/v2/train.jsonl --limit 30 --out {OUT}/train30.jsonl
!python colab/predict.py {A} --records data/dataset/v2/dev.jsonl --out {OUT}/dev.jsonl
!python colab/predict.py {A} --records data/human_test/v1/paraphrase.jsonl --out {OUT}/paraphrase.jsonl
drive_free(0.2)

### 8б. Остальные выборки (для итоговой таблицы)
test v2 (500), `gen_*` (300), `mc12_real` (13): около 800 запросов. Нужны для моделей, которые не помещаются на ноутбук автора (9B); остальные можно считать локально. Каждый файл продолжается с места обрыва: если сессия кончится, запустить ячейки 1-3 и эту снова.

In [ ]:
A = f'--model {MODEL} --adapter {OUT}/adapter --format {FORMAT} --name {RUN}'
!python colab/predict.py {A} --records data/dataset/mc12_real_v1/mc12_real.jsonl --out {OUT}/mc12_real.jsonl
for s in ['gen_topology', 'gen_family', 'gen_range', 'test']:
    !python colab/predict.py {A} --records data/dataset/v2/{s}.jsonl --out {OUT}/{s}.jsonl
drive_free(0.2)

## 9. Что получилось
Здесь быстрая прикидка: точное совпадение строк ответа с эталоном, без учёта порядка. Настоящая оценка с MC12 идёт отдельно, на ноутбуке с Micro-Cap: она считает схему верной и при других именах узлов, поэтому её цифры выше.

In [ ]:
import sys
sys.path.insert(0, 'src')
from mcnl import training
from IPython.display import display, Markdown


def lines(text):
    return sorted(l.strip() for l in (text or '').splitlines() if l.strip())


def show(title, records, answers, n=4):
    gold = {r['id']: r for r in training.read_jsonl(records)}
    rows = training.read_jsonl(answers)
    same = sum(lines(a['output']) == lines(training.answer_for(gold[a['id']], FORMAT)) for a in rows)
    display(Markdown(f'### {title}: совпало строка в строку {same} из {len(rows)}'))
    for a in rows[:n]:
        g = gold[a['id']]
        print('ЗАПРОС:', g['prompt'][:200]); print('ЭТАЛОН:', training.answer_for(g, FORMAT).replace(chr(10), ' | '))
        print('ОТВЕТ: ', a['output'].replace(chr(10), ' | ')[:300]); print()


show('Без обучения, dev', 'data/dataset/v2/dev.jsonl', f'{OUT}/untrained_dev.jsonl')
show('Обучающая выборка', 'data/dataset/v2/train.jsonl', f'{OUT}/train30.jsonl')
show('Обученная модель, dev', 'data/dataset/v2/dev.jsonl', f'{OUT}/dev.jsonl')
show('Запросы C01', 'data/human_test/v1/paraphrase.jsonl', f'{OUT}/paraphrase.jsonl')

## 10. Дописать на Drive
Colab отправляет файлы на Drive не сразу. Перед закрытием сессии - эта ячейка: она дожидается отправки и отключает Drive. Ниже - что лежит в папке запуска; эти же файлы должны быть видны на drive.google.com.

In [ ]:
!ls -la {OUT}
drive_free(0)
drive.flush_and_unmount()
print('Drive отключён, файлы отправлены. Для новых ячеек - снова ячейка 2.')